# Medical Necessity — Documentation Coach (RAG + RL)

Turns the medical necessity scoring work from a **classifier** into a **coach**: instead of
labelling an order after the fact, it identifies which CMS-relevant clinical elements are missing
from the order text and what to ask the ordering clinician for.

**Three layers**

1. **CMS knowledge layer** — the elements CMS looks for, keyed to the source that requires them.
2. **Narrative analysis** — which of those elements the order text actually documents.
3. **Recommendation policy** — which missing element to request next.

**On the reinforcement learning layer.** The reward function depends on downstream outcomes
(approved first pass, documentation-related denial, approved after appeal). That outcome data is
not yet available — it is the MUSC and excess-health denial extract still pending. Until it lands,
the agent cannot be trained.

This notebook therefore builds the full state / action / reward structure and runs the policy on an
**information-gain prior**: recommend the element that most often distinguishes a well-documented
order from a poorly-documented one. Section 10 is where the learned reward replaces the prior, and
it is a single function swap. Everything before that is usable today.

## 1. Configuration

In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from datetime import datetime
from pyspark.sql import functions as F

SCORED_TABLE = "`prod-sandbox`.`josh_smitherman`.`med_nec_genie`"
OUTPUT_DIR   = "/Workspace/Users/josh.smitherman@gmr.net/med_nec/coach"
RUN_TS       = datetime.now().strftime("%Y%m%d_%H%M%S")

LLM_MODEL    = "databricks-gpt-oss-120b"
LLM_BASE_URL = "https://adb-2790612761746757.17.azuredatabricks.net/serving-endpoints"
LLM_SAMPLE_N = 500
LLM_WORKERS  = 16
LLM_SEED     = 42

OUTCOMES_TABLE = None

matplotlib.rcParams.update({
    "figure.dpi": 110,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "-",
})
GREY = "#4d4d4d"
LGREY = "#b0b0b0"

os.makedirs("/tmp/coach", exist_ok=True)
print(f"Run {RUN_TS}")
print(f"Scored table: {SCORED_TABLE}")
print(f"Outcomes table: {OUTCOMES_TABLE or 'NOT AVAILABLE - policy runs on prior'}")

## 2. CMS knowledge layer

The elements a reviewer looks for, each tied to the CMS text that requires it and to the concept columns already present in the scored table.

In [ ]:
CMS_ELEMENTS = [
    {"element": "mobility_limitation",
     "question": "What is the patient unable to do — bear weight, ambulate, sit upright?",
     "axis": "mobility",
     "cms_ref": "BPM10 10.2.3",
     "concepts": ["bed_confined", "mobility_deficit", "cannot_sit"],
     "named": True},
    {"element": "alternate_transport_contraindicated",
     "question": "Why is a wheelchair van or private vehicle unsafe for this patient?",
     "axis": "mobility",
     "cms_ref": "BPM10 10.2.1",
     "concepts": ["bed_confined", "cannot_sit", "bariatric", "wound_ostomy"],
     "named": True},
    {"element": "handling_requirement",
     "question": "Does the patient need special handling, positioning or lift assistance?",
     "axis": "mobility",
     "cms_ref": "BPM10 10.2.1",
     "concepts": ["bariatric", "wound_ostomy"],
     "named": False},
    {"element": "behavioral_risk",
     "question": "Is there an altered mental status or behavioral risk during transport?",
     "axis": "mobility",
     "cms_ref": "BPM10 10.2.1",
     "concepts": ["behavioral"],
     "named": False},
    {"element": "airway_support",
     "question": "Does the patient need ventilator support, an artificial airway or suctioning?",
     "axis": "monitoring",
     "cms_ref": "414.605",
     "concepts": ["ventilator", "suctioning"],
     "named": True},
    {"element": "medication_administration",
     "question": "Are IV medications or infusions required during transport?",
     "axis": "monitoring",
     "cms_ref": "414.605",
     "concepts": ["iv_medication"],
     "named": True},
    {"element": "cardiac_monitoring",
     "question": "Is cardiac monitoring or ongoing assessment required en route?",
     "axis": "monitoring",
     "cms_ref": "414.605",
     "concepts": ["cardiac"],
     "named": True},
    {"element": "oxygen_requirement",
     "question": "Is supplemental oxygen required, and at what flow?",
     "axis": "monitoring",
     "cms_ref": "BPM10 10.2.1",
     "concepts": ["oxygen"],
     "named": False},
    {"element": "isolation_precautions",
     "question": "Are isolation or infection-control precautions in effect?",
     "axis": "monitoring",
     "cms_ref": "BPM10 10.2.1",
     "concepts": ["isolation"],
     "named": False},
]

ELEMENTS = [e["element"] for e in CMS_ELEMENTS]
ELEMENT_META = {e["element"]: e for e in CMS_ELEMENTS}
NAMED_ELEMENTS = [e["element"] for e in CMS_ELEMENTS if e["named"]]

knowledge = pd.DataFrame([{
    "element": e["element"],
    "axis": e["axis"],
    "named_by_cms": e["named"],
    "cms_ref": e["cms_ref"],
    "concepts": ", ".join(e["concepts"]),
    "question": e["question"],
} for e in CMS_ELEMENTS])
display(knowledge)

## 3. Load the scored orders

In [ ]:
scored = spark.table(SCORED_TABLE)
available = set(scored.columns)

id_col = next((c for c in ["TripLegId", "TripRequestId", "OrderId"] if c in available), None)
los_col = next((c for c in ["LevelOfService", "LOS", "ServiceLevel"] if c in available), None)
cust_col = next((c for c in ["Customer", "CustomerName", "ContractName", "Facility"] if c in available), None)

concept_cols = sorted({c for e in CMS_ELEMENTS for c in e["concepts"]} & available)
keep = [c for c in [id_col, los_col, cust_col] if c] + concept_cols + [
    c for c in ["necessity_class", "total_score", "named_score", "has_named_concept",
                "unmatched_text", "clinical_text"] if c in available
]

orders = scored.select(*keep).toPandas()
for c in concept_cols:
    orders[c] = pd.to_numeric(orders[c], errors="coerce").fillna(0).astype(int)

print(f"Orders loaded: {len(orders):,}")
print(f"Identifier: {id_col} | Level of service: {los_col} | Customer: {cust_col}")
print(f"Concept columns found: {len(concept_cols)} of {len({c for e in CMS_ELEMENTS for c in e['concepts']})}")
missing_concepts = {c for e in CMS_ELEMENTS for c in e["concepts"]} - available
if missing_concepts:
    print(f"NOT IN TABLE (elements using them will always read as missing): {sorted(missing_concepts)}")

## 4. Element coverage and gap detection

An element is *documented* when any concept mapped to it appears in the order text, and a *gap* otherwise.

In [ ]:
for e in CMS_ELEMENTS:
    cols = [c for c in e["concepts"] if c in orders.columns]
    if cols:
        orders[f"doc_{e['element']}"] = (orders[cols].sum(axis=1) > 0).astype(int)
    else:
        orders[f"doc_{e['element']}"] = 0
    orders[f"gap_{e['element']}"] = 1 - orders[f"doc_{e['element']}"]

doc_cols = [f"doc_{e}" for e in ELEMENTS]
gap_cols = [f"gap_{e}" for e in ELEMENTS]

orders["elements_documented"] = orders[doc_cols].sum(axis=1)
orders["elements_missing"] = orders[gap_cols].sum(axis=1)
orders["mobility_documented"] = orders[[f"doc_{e['element']}" for e in CMS_ELEMENTS if e["axis"] == "mobility"]].max(axis=1)
orders["monitoring_documented"] = orders[[f"doc_{e['element']}" for e in CMS_ELEMENTS if e["axis"] == "monitoring"]].max(axis=1)
orders["both_axes_documented"] = ((orders.mobility_documented == 1) & (orders.monitoring_documented == 1)).astype(int)
orders["named_documented"] = orders[[f"doc_{e}" for e in NAMED_ELEMENTS]].max(axis=1)

n = len(orders)
summary = pd.DataFrame([
    ["Orders in scope", n, "100.0%"],
    ["No element documented", int((orders.elements_documented == 0).sum()),
     f"{(orders.elements_documented == 0).mean():.1%}"],
    ["Mobility axis documented", int(orders.mobility_documented.sum()),
     f"{orders.mobility_documented.mean():.1%}"],
    ["Monitoring axis documented", int(orders.monitoring_documented.sum()),
     f"{orders.monitoring_documented.mean():.1%}"],
    ["Both axes documented", int(orders.both_axes_documented.sum()),
     f"{orders.both_axes_documented.mean():.1%}"],
    ["A CMS-named element documented", int(orders.named_documented.sum()),
     f"{orders.named_documented.mean():.1%}"],
], columns=["measure", "orders", "share"])
display(summary)

## 5. Where the documentation gaps are

In [ ]:
gap_rates = (pd.DataFrame({
        "element": ELEMENTS,
        "gap_rate": [orders[f"gap_{e}"].mean() for e in ELEMENTS],
        "gap_orders": [int(orders[f"gap_{e}"].sum()) for e in ELEMENTS],
        "axis": [ELEMENT_META[e]["axis"] for e in ELEMENTS],
        "named_by_cms": [ELEMENT_META[e]["named"] for e in ELEMENTS],
    })
    .sort_values("gap_rate", ascending=True)
    .reset_index(drop=True))

fig, ax = plt.subplots(figsize=(8.5, 4.2))
colors = [GREY if nm else LGREY for nm in gap_rates.named_by_cms]
ax.barh(gap_rates.element, gap_rates.gap_rate * 100, color=colors, height=0.68)
for i, (r, c) in enumerate(zip(gap_rates.gap_rate, gap_rates.gap_orders)):
    ax.text(r * 100 + 0.8, i, f"{r:.0%}  ({c:,})", va="center", fontsize=8)
ax.set_xlabel("Share of orders where the element is not documented (%)")
ax.set_xlim(0, 108)
ax.set_title("Documentation gap rate by CMS element\ndark = named explicitly by CMS, light = inferred", loc="left")
plt.tight_layout(); plt.show()

display(gap_rates.sort_values("gap_rate", ascending=False).reset_index(drop=True))

## 6. How many elements a single order is missing

In [ ]:
dist = orders.elements_missing.value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].bar(dist.index, dist.values, color=GREY, width=0.7)
axes[0].set_xlabel(f"Elements missing (of {len(ELEMENTS)})")
axes[0].set_ylabel("Orders")
axes[0].set_title("Missing elements per order", loc="left")
for x, y in zip(dist.index, dist.values):
    axes[0].text(x, y, f"{y/n:.0%}", ha="center", va="bottom", fontsize=7)

by_class = (orders.groupby("necessity_class").elements_missing.mean().sort_values())
axes[1].barh(by_class.index, by_class.values, color=GREY, height=0.6)
for i, v in enumerate(by_class.values):
    axes[1].text(v + 0.05, i, f"{v:.1f}", va="center", fontsize=8)
axes[1].set_xlabel("Mean elements missing")
axes[1].set_title("Missing elements by necessity class", loc="left")
axes[1].set_xlim(0, len(ELEMENTS))
plt.tight_layout(); plt.show()

## 7. Which gaps travel together

Co-occurrence tells the policy whether one question can close several gaps at once.

In [ ]:
G = orders[gap_cols].values
co = (G.T @ G) / np.maximum(G.sum(axis=0)[:, None], 1)
labels = [c.replace("gap_", "") for c in gap_cols]

fig, ax = plt.subplots(figsize=(7.4, 6.2))
im = ax.imshow(co, cmap="Greys", vmin=0, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
ax.set_title("P(column also missing | row missing)", loc="left")
ax.grid(False)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{co[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if co[i, j] > 0.6 else "black")
fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout(); plt.show()

## 8. Gap patterns by facility and level of service

In [ ]:
if cust_col:
    top_cust = orders[cust_col].value_counts().head(12).index
    by_cust = (orders[orders[cust_col].isin(top_cust)]
               .groupby(cust_col)[["elements_missing", "named_documented", "both_axes_documented"]]
               .agg({"elements_missing": "mean", "named_documented": "mean", "both_axes_documented": "mean"})
               .sort_values("elements_missing"))
    volume = orders[orders[cust_col].isin(top_cust)][cust_col].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
    axes[0].barh(by_cust.index, by_cust.elements_missing, color=GREY, height=0.62)
    for i, (v, name) in enumerate(zip(by_cust.elements_missing, by_cust.index)):
        axes[0].text(v + 0.05, i, f"{v:.1f}  (n={volume[name]:,})", va="center", fontsize=7.5)
    axes[0].set_xlabel("Mean elements missing")
    axes[0].set_xlim(0, len(ELEMENTS))
    axes[0].set_title("Mean documentation gaps by facility", loc="left")

    axes[1].barh(by_cust.index, by_cust.named_documented * 100, color=GREY, height=0.62)
    for i, v in enumerate(by_cust.named_documented):
        axes[1].text(v * 100 + 0.8, i, f"{v:.0%}", va="center", fontsize=7.5)
    axes[1].set_xlabel("Orders documenting a CMS-named element (%)")
    axes[1].set_xlim(0, 108)
    axes[1].set_title("Named-element coverage by facility", loc="left")
    plt.tight_layout(); plt.show()
    display(by_cust.round(3))

if los_col:
    by_los = (orders.groupby(los_col)
              .agg(orders_n=("elements_missing", "size"),
                   mean_missing=("elements_missing", "mean"),
                   named_rate=("named_documented", "mean"))
              .query("orders_n >= 100")
              .sort_values("mean_missing"))
    fig, ax = plt.subplots(figsize=(8.5, 3.6))
    ax.bar(by_los.index.astype(str), by_los.mean_missing, color=GREY, width=0.6)
    for i, (v, cnt) in enumerate(zip(by_los.mean_missing, by_los.orders_n)):
        ax.text(i, v, f"{v:.1f}\nn={cnt:,}", ha="center", va="bottom", fontsize=7)
    ax.set_ylabel("Mean elements missing")
    ax.set_ylim(0, len(ELEMENTS))
    ax.set_title("Documentation gaps by level of service", loc="left")
    plt.tight_layout(); plt.show()

## 9. Narrative analysis on a sample

The keyword mapping above cannot read phrasing it was not written for. This layer extracts the same elements with a language model, and records which ones it judged absent.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

_token = (dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
          if "dbutils" in dir() else os.environ.get("DATABRICKS_TOKEN", ""))
_client = OpenAI(api_key=_token, base_url=LLM_BASE_URL)

COACH_PROMPT = """
You are a clinical documentation extraction assistant for non-emergent ground ambulance orders.

Read the transport reason recorded at order time and report, for each element below, whether the
text documents it. Do not decide whether the transport was necessary or appropriate. Do not give
medical advice. Report only what the text supports; handle negations correctly.

ELEMENTS
mobility_limitation: what the patient cannot do - bear weight, ambulate, sit upright
alternate_transport_contraindicated: why a wheelchair van or private vehicle is unsafe
handling_requirement: special handling, positioning or lift assistance
behavioral_risk: altered mental status or behavioral risk in transport
airway_support: ventilator, artificial airway or suctioning
medication_administration: IV medications or infusions in transport
cardiac_monitoring: cardiac monitoring or ongoing assessment en route
oxygen_requirement: supplemental oxygen
isolation_precautions: isolation or infection-control precautions

Return valid JSON only, no preamble:
{
  "documented": {"<element>": true|false, ... all nine ...},
  "evidence": [{"element": "<element>", "quote": "<verbatim text>"}],
  "absent_summary": "<one or two sentences naming what the text does not establish>"
}

Every element set to true must have a verbatim quote in evidence.
"""

def extract_one(rec):
    oid, text = rec
    out = {"_order_id": oid, "llm_ok": 0}
    try:
        resp = _client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "system", "content": COACH_PROMPT},
                      {"role": "user", "content": f"Order text:\n{text or ''}"}],
            temperature=0.1, max_tokens=1200,
        )
        content = resp.choices[0].message.content
        if isinstance(content, list):
            content = next((i.get("text", "") for i in content
                            if isinstance(i, dict) and i.get("type") == "text"), "")
        obj = json.loads(content)
        doc = obj.get("documented", {})
        if not all(isinstance(doc.get(e), bool) for e in ELEMENTS):
            raise ValueError("missing or non-boolean element")
        for e in ELEMENTS:
            out[f"llm_doc_{e}"] = int(doc[e])
        out["llm_absent_summary"] = obj.get("absent_summary", "")
        out["llm_evidence"] = "; ".join(
            f"{d.get('element')}[{d.get('quote')}]" for d in obj.get("evidence", [])
            if isinstance(d, dict))
        out["llm_ok"] = 1
    except Exception as exc:
        out["llm_error"] = str(exc)[:200]
    return out

sample = orders[orders.clinical_text.notna() & (orders.clinical_text.str.strip() != "")]
sample = sample.sample(n=min(LLM_SAMPLE_N, len(sample)), random_state=LLM_SEED)
print(f"Narrative analysis on {len(sample):,} orders (seed {LLM_SEED}).")

with ThreadPoolExecutor(max_workers=LLM_WORKERS) as pool:
    llm_rows = list(pool.map(extract_one, zip(sample[id_col], sample.clinical_text)))

llm_df = pd.DataFrame(llm_rows)
ok_rate = llm_df.llm_ok.mean()
print(f"Valid extractions: {ok_rate:.1%}")

llm_ok = llm_df[llm_df.llm_ok == 1]
if len(llm_ok):
    merged = sample.merge(llm_ok, left_on=id_col, right_on="_order_id", how="inner")
    compare = pd.DataFrame({
        "element": ELEMENTS,
        "keyword_documented": [merged[f"doc_{e}"].mean() for e in ELEMENTS],
        "llm_documented": [merged[f"llm_doc_{e}"].mean() for e in ELEMENTS],
    })
    compare["difference"] = compare.llm_documented - compare.keyword_documented

    fig, ax = plt.subplots(figsize=(9, 4.2))
    y = np.arange(len(compare))
    ax.barh(y - 0.2, compare.keyword_documented * 100, height=0.38, color=LGREY, label="keyword rules")
    ax.barh(y + 0.2, compare.llm_documented * 100, height=0.38, color=GREY, label="language model")
    ax.set_yticks(y); ax.set_yticklabels(compare.element, fontsize=8)
    ax.set_xlabel("Share of sampled orders where the element is documented (%)")
    ax.set_title("What each reading finds in the same text", loc="left")
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.show()
    display(compare.round(3))

## 10. Reward layer

The reward function needs downstream outcomes per order. Set `OUTCOMES_TABLE` in section 1 once the
denial extract is available; the schema needed is one row per order with an outcome in
`approved_first_pass`, `clarification_requested`, `approved_after_appeal`, `documentation_denial`,
`missing_clinical_justification`.

Until then `REWARDS` is empty and the policy in section 11 falls back to the information-gain prior.

In [ ]:
REWARD_SCALE = {
    "approved_first_pass": 20,
    "clarification_requested": -10,
    "approved_after_appeal": 5,
    "documentation_denial": -20,
    "missing_clinical_justification": -25,
    "query_to_clinician": -2,
}

REWARDS = None
HAS_OUTCOMES = False

if OUTCOMES_TABLE:
    outcomes = spark.table(OUTCOMES_TABLE).toPandas()
    joined = orders.merge(outcomes, left_on=id_col, right_on=id_col, how="inner")
    joined["reward"] = joined["outcome"].map(REWARD_SCALE)
    HAS_OUTCOMES = joined["reward"].notna().sum() > 0
    if HAS_OUTCOMES:
        REWARDS = joined
        print(f"Outcomes joined: {len(joined):,} orders, "
              f"{joined.reward.notna().sum():,} with a mapped reward.")
        rate = (joined.groupby("outcome").size() / len(joined)).sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(7.5, 3.2))
        ax.barh(rate.index, rate.values * 100, color=GREY, height=0.6)
        for i, v in enumerate(rate.values):
            ax.text(v * 100 + 0.6, i, f"{v:.1%}", va="center", fontsize=8)
        ax.set_xlabel("Share of orders (%)")
        ax.set_title("Outcome distribution", loc="left")
        plt.tight_layout(); plt.show()

if not HAS_OUTCOMES:
    print("No outcome data. The policy will run on the information-gain prior.")
    print("This means recommendations are ranked by how strongly an element separates")
    print("well-documented orders from poorly-documented ones - NOT by denial impact.")

## 11. Recommendation policy

**State** — which elements the order documents, plus level of service and facility.
**Action** — request one missing element.
**Value** — expected gain from asking.

With outcomes, value is the observed reward difference between orders that documented the element
and those that did not. That comparison is against an external signal, so it is a real measurement.

Without outcomes the prior is stated directly rather than fitted, because fitting it against the
scored data is circular: documenting a CMS-named element *is* what makes an order carry a named
justification, so any regression on that target recovers the definition instead of a finding. The
prior scores an ask by what closing it would supply:

- **1.0** if CMS names the element explicitly — closing it directly supplies a named justification
- **0.3** if the element is inferred — it supports a case but cannot carry one alone
- **+0.5** if that element's axis is currently undocumented for this order — CMS applies both the
  mobility and the monitoring test, so covering an empty axis is worth more than deepening a
  covered one

These three numbers are assumptions, not measurements. They are what the denial data replaces.

In [ ]:
VALUE_NAMED = 1.0
VALUE_INFERRED = 0.3
VALUE_UNCOVERED_AXIS = 0.5

def ask_value(element, order_row):
    meta = ELEMENT_META[element]
    v = VALUE_NAMED if meta["named"] else VALUE_INFERRED
    axis_flag = "mobility_documented" if meta["axis"] == "mobility" else "monitoring_documented"
    if order_row.get(axis_flag, 0) == 0:
        v += VALUE_UNCOVERED_AXIS
    return v

def prior_policy(df):
    rows = []
    for e in ELEMENTS:
        gap_rows = df[df[f"gap_{e}"] == 1]
        if not len(gap_rows):
            rows.append({"element": e, "value": 0.0, "basis": "no gaps",
                         "gap_orders": 0, "prevalence_of_gap": 0.0, "expected_gain": 0.0})
            continue
        meta = ELEMENT_META[e]
        axis_flag = "mobility_documented" if meta["axis"] == "mobility" else "monitoring_documented"
        base = VALUE_NAMED if meta["named"] else VALUE_INFERRED
        mean_value = base + VALUE_UNCOVERED_AXIS * (gap_rows[axis_flag] == 0).mean()
        prevalence = df[f"gap_{e}"].mean()
        rows.append({"element": e,
                     "value": float(mean_value),
                     "basis": "CMS-weighted prior",
                     "gap_orders": int(len(gap_rows)),
                     "prevalence_of_gap": float(prevalence),
                     "expected_gain": float(mean_value * prevalence)})
    return pd.DataFrame(rows).sort_values("expected_gain", ascending=False).reset_index(drop=True)

def learned_policy(df):
    rows = []
    for e in ELEMENTS:
        has = df[(df[f"doc_{e}"] == 1) & df.reward.notna()]
        lacks = df[(df[f"doc_{e}"] == 0) & df.reward.notna()]
        prevalence = df[f"gap_{e}"].mean()
        if len(has) < 30 or len(lacks) < 30:
            rows.append({"element": e, "value": 0.0, "basis": "insufficient outcomes",
                         "gap_orders": int(len(lacks)), "prevalence_of_gap": float(prevalence),
                         "expected_gain": 0.0})
            continue
        lift = float(has.reward.mean() - lacks.reward.mean())
        gain = lift * prevalence + REWARD_SCALE["query_to_clinician"]
        rows.append({"element": e, "value": lift, "basis": "observed reward",
                     "gap_orders": int(len(lacks)), "prevalence_of_gap": float(prevalence),
                     "expected_gain": gain})
    return pd.DataFrame(rows).sort_values("expected_gain", ascending=False).reset_index(drop=True)

policy = learned_policy(REWARDS) if HAS_OUTCOMES else prior_policy(orders)
BASIS = "observed reward" if HAS_OUTCOMES else "CMS-weighted prior"

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
p = policy.sort_values("expected_gain")
bar_colors = [GREY if ELEMENT_META[e]["named"] else LGREY for e in p.element]
axes[0].barh(p.element, p.expected_gain, color=bar_colors, height=0.62)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel(f"Expected gain from requesting ({BASIS})")
axes[0].set_title("Action value — which gap to ask about first\ndark = named by CMS, light = inferred", loc="left")

axes[1].scatter(policy.prevalence_of_gap * 100, policy.value, s=44,
                color=[GREY if ELEMENT_META[e]["named"] else LGREY for e in policy.element])
for _, r in policy.iterrows():
    axes[1].annotate(r.element, (r.prevalence_of_gap * 100, r.value),
                     fontsize=7, xytext=(4, 3), textcoords="offset points")
axes[1].set_xlabel("How often the gap occurs (%)")
axes[1].set_ylabel("Value when closed")
axes[1].set_title("Frequency against value\ntop right = ask about these first", loc="left")
plt.tight_layout(); plt.show()

display(policy.round(4))

## 12. Coaching output for a single order

What an ordering clinician would see at point of entry.

In [ ]:
def coach(order_row, max_asks=3):
    gaps = [e for e in ELEMENTS if order_row.get(f"gap_{e}", 1) == 1]
    if HAS_OUTCOMES:
        order_rank = {e: float(policy.loc[policy.element == e, "expected_gain"].iloc[0])
                      for e in gaps}
    else:
        order_rank = {e: ask_value(e, order_row) for e in gaps}
    ranked = sorted(gaps, key=lambda e: -order_rank[e])[:max_asks]
    return {
        "order_id": order_row.get(id_col),
        "text": order_row.get("clinical_text"),
        "elements_documented": [e for e in ELEMENTS if order_row.get(f"doc_{e}") == 1],
        "elements_missing": gaps,
        "requests": [{
            "element": e,
            "ask": ELEMENT_META[e]["question"],
            "cms_ref": ELEMENT_META[e]["cms_ref"],
            "named_by_cms": ELEMENT_META[e]["named"],
            "value": float(order_rank[e]),
        } for e in ranked],
    }

def render(c):
    print("=" * 78)
    print(f"Order {c['order_id']}")
    print(f"Text: {c['text']}")
    print(f"\nDocumented ({len(c['elements_documented'])}): "
          f"{', '.join(c['elements_documented']) or 'nothing'}")
    print(f"Missing ({len(c['elements_missing'])}): {', '.join(c['elements_missing'])}")
    if c["requests"]:
        print("\nSuggested clarifications, highest value first:")
        for i, r in enumerate(c["requests"], 1):
            tag = "CMS-named" if r["named_by_cms"] else "inferred"
            print(f"  {i}. {r['ask']}")
            print(f"     {r['cms_ref']} | {tag} | value {r['value']:.2f}")
    print(f"\nBasis: {BASIS}. Guidance on documentation completeness, not a")
    print("necessity or billing determination.")

worst = orders.sort_values("elements_missing", ascending=False)
worst = worst[worst.clinical_text.notna() & (worst.clinical_text.str.strip() != "")]
for _, row in worst.head(3).iterrows():
    render(coach(row))

## 13. What the coach would ask for across the population

In [ ]:
ask_counts = {e: 0 for e in ELEMENTS}
for _, row in orders.iterrows():
    gaps = [e for e in ELEMENTS if row.get(f"gap_{e}", 1) == 1]
    for e in sorted(gaps, key=lambda x: -ask_value(x, row))[:3]:
        ask_counts[e] += 1

asks = (pd.Series(ask_counts).sort_values(ascending=True) / len(orders))
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
axes[0].barh(asks.index, asks.values * 100, color=GREY, height=0.62)
for i, v in enumerate(asks.values):
    axes[0].text(v * 100 + 0.7, i, f"{v:.0%}", va="center", fontsize=8)
axes[0].set_xlabel("Share of orders where this would be asked (%)")
axes[0].set_xlim(0, 108)
axes[0].set_title("Clarification volume by element\ncapped at three asks per order", loc="left")

reachable = orders.copy()
for e in ELEMENTS:
    reachable[f"after_{e}"] = reachable[[f"doc_{x}" for x in NAMED_ELEMENTS]].max(axis=1)
steps = pd.Series({
    "documents a named element now": orders.named_documented.mean(),
    "documents any element now": (orders.elements_documented > 0).mean(),
    "documents both axes now": orders.both_axes_documented.mean(),
    "documents nothing now": (orders.elements_documented == 0).mean(),
}).sort_values()
axes[1].barh(steps.index, steps.values * 100, color=GREY, height=0.6)
for i, v in enumerate(steps.values):
    axes[1].text(v * 100 + 0.7, i, f"{v:.0%}", va="center", fontsize=8)
axes[1].set_xlabel("Share of orders (%)")
axes[1].set_xlim(0, 108)
axes[1].set_title("Current documentation state", loc="left")
plt.tight_layout(); plt.show()

## 14. Write outputs

In [ ]:
import shutil

gap_detail = orders[[c for c in [id_col, los_col, cust_col] if c] + doc_cols + gap_cols +
                    ["elements_documented", "elements_missing", "mobility_documented",
                     "monitoring_documented", "both_axes_documented", "named_documented",
                     "necessity_class", "clinical_text"]]

files = {}
local = f"/tmp/coach/med_nec_gap_detail_{RUN_TS}.csv"
gap_detail.to_csv(local, index=False)
files["gap detail"] = local

local_policy = f"/tmp/coach/med_nec_policy_{RUN_TS}.csv"
policy.assign(basis_of_run=BASIS).to_csv(local_policy, index=False)
files["policy"] = local_policy

local_knowledge = f"/tmp/coach/med_nec_cms_elements_{RUN_TS}.csv"
knowledge.to_csv(local_knowledge, index=False)
files["cms elements"] = local_knowledge

os.makedirs(OUTPUT_DIR, exist_ok=True)
for label, path in files.items():
    dest = os.path.join(OUTPUT_DIR, os.path.basename(path))
    try:
        shutil.copyfile(path, dest)
        print(f"{label}: {dest}")
    except Exception as exc:
        print(f"{label}: wrote {path}, copy failed - {exc}")

print(f"\nPolicy basis this run: {BASIS}")
if not HAS_OUTCOMES:
    print("Set OUTCOMES_TABLE once the denial extract lands to switch to learned rewards.")

## Limits

- **The RL layer is not trained.** Without outcome data the policy ranks gaps by information gain,
  which says which elements distinguish well-documented orders from poorly-documented ones. It does
  **not** say which gaps cause denials. Those can differ.
- **Element coverage is inferred from keyword concepts** in sections 4 to 8. Roughly one order in
  eight contains text no keyword rule reads, so gap rates are an upper bound on what is genuinely
  missing. Section 9 measures that difference on a sample.
- **Order-time documentation only.** The crew patient care report is in ImageTrend and is not here.
  A gap at order time may be closed by the PCR.
- **Guidance, not adjudication.** Output describes documentation completeness. It is not a
  necessity determination, a coverage decision, or advice about a specific patient.
- **A clarification has a cost.** Each request interrupts a clinician. The three-ask cap and the
  query penalty in `REWARD_SCALE` exist for that reason; both need a real number from operations
  before this reaches point of entry.